In [2]:
pip install librosa soundfile numpy scikit-learn pyaudio

  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl.metadata (16 kB)
  Using cached PyAudio-0.2.14-cp311-cp311-win_amd64.whl.metadata (2.7 kB)
   ---------------------------------------- 0.0/260.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/260.7 kB ? eta -:--:--
   --------- ------------------------------ 61.4/260.7 kB 1.1 MB/s eta 0:00:01
   ------------------------------------ --- 235.5/260.7 kB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 260.7/260.7 kB 1.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------- ----------------- 0.6/1.0 MB 18.5 MB/s eta 0:00:01
   ---------------------------------------  1.0/1.0 MB 16.2 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 10.8 MB/s eta 0:00:00
Using cached PyAudio-0.2.14-cp311-cp311-win_amd64.whl (164 kB)
   -------------------------------------


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: C:\Users\Betina Kuriakose\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import librosa
import soundfile
import os, glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [22]:
def extract_feature(file_name, mfcc, chroma, mel):
    try:
        with soundfile.SoundFile(file_name) as sound_file:
            X = sound_file.read(dtype="float32")
            sample_rate = sound_file.samplerate

            # skip very short audio
            if len(X) < 2048:
                return None

            result = np.array([])

            if chroma:
                stft = np.abs(librosa.stft(X))

            if mfcc:
                mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40), axis=1)
                result = np.hstack((result, mfccs))

            if chroma:
                chroma_feat = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate), axis=1)
                result = np.hstack((result, chroma_feat))

            if mel:
                mel_feat = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate), axis=1)
                result = np.hstack((result, mel_feat))

        return result

    except Exception:
        return None

In [23]:
#DataFlair - Emotions in the RAVDESS dataset
emotions={
  '01':'neutral',
  '02':'calm',
  '03':'happy',
  '04':'sad',
  '05':'angry',
  '06':'fearful',
  '07':'disgust',
  '08':'surprised'
}

#DataFlair - Emotions to observe
observed_emotions=['calm', 'happy', 'fearful', 'disgust']

In [ ]:
C:\\Users\\Betina Kuriakose\\Desktop\\Speech_Emotion\\data\\dataset\\Actor_*\\*.wav

In [26]:
def load_data(test_size=0.2):
    x, y = [], []

    for file in glob.glob("C:\\Users\\Betina Kuriakose\\Desktop\\Speech_Emotion\\data\\dataset\\Actor_*\\*.wav"):
        file_name = os.path.basename(file)
        emotion = emotions[file_name.split("-")[2]]

        if emotion not in observed_emotions:
            continue

        feature = extract_feature(file, mfcc=True, chroma=True, mel=True)

        if feature is None:
            continue

        if len(feature) == 180:   # expected size (40 MFCC + 12 chroma + 128 mel)
            x.append(feature)
            y.append(emotion)

    return train_test_split(np.array(x), y, test_size=test_size, random_state=9)

In [27]:
#DataFlair - Split the dataset
x_train,x_test,y_train,y_test=load_data(test_size=0.25)

In [28]:
#DataFlair - Get the shape of the training and testing datasets
print((x_train.shape[0], x_test.shape[0]))

(573, 191)


In [29]:
#DataFlair - Get the number of features extracted
print(f'Features extracted: {x_train.shape[1]}')

Features extracted: 180


In [30]:
#DataFlair - Initialize the Multi Layer Perceptron Classifier
model=MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08, hidden_layer_sizes=(300,), learning_rate='adaptive', max_iter=500)

In [31]:
#DataFlair - Train the model
model.fit(x_train,y_train)

MLPClassifier(alpha=0.01, batch_size=256, hidden_layer_sizes=(300,),
              learning_rate='adaptive', max_iter=500)

In [32]:
#DataFlair - Predict for the test set
y_pred=model.predict(x_test)

In [33]:
#DataFlair - Calculate the accuracy of our model
accuracy=accuracy_score(y_true=y_test, y_pred=y_pred)

#DataFlair - Print the accuracy
print("Accuracy: {:.2f}%".format(accuracy*100))

Accuracy: 73.30%
